# Annotated tracking benchmark

Compare local GMM tracking with a static initial box on VOT and TrackingDataset. Recorded summaries are retained separately from cells that execute a benchmark.


## 1. Local search

Initialise from the first annotation and search around the previous prediction. Later annotations are used only for scoring.


In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import evaluate as ev
import tracker
from tonic_convert import DVSBenchmark

warnings.filterwarnings("ignore")

VOT = "data/DVSBENCH/INI_VOT_30fps_20160610.hdf5"
TRACK = "data/DVSBENCH/INI_TrackingDataset_30fps_20160610.hdf5"
WINDOW = 33_333

vot = DVSBenchmark(VOT)
track = DVSBenchmark(TRACK)
print("VOT sequences            :", len(vot.data))
print("TrackingDataset sequences:", len(track.data))
print("search window            :", tracker.SEARCH_SCALE, "x the current box")
print("frame window             :", WINDOW / 1000, "ms  (one 30 fps video frame)")

## 2. Target motion

Measure target displacement relative to box size and the overlap of the static baseline. This describes the difficulty of each sequence.


In [ ]:
rows = []
for nm in vot.data:
    boxes = np.array([ev.gt_box(r) for r in vot._boxes[nm]])
    cx = (boxes[:, 0] + boxes[:, 2]) / 2
    cy = (boxes[:, 1] + boxes[:, 3]) / 2
    diag = max(
        np.hypot(np.median(boxes[:, 2] - boxes[:, 0]), np.median(boxes[:, 3] - boxes[:, 1])), 1
    )
    step = np.hypot(np.diff(cx), np.diff(cy))
    static = np.mean([ev.iou(tuple(boxes[0]), tuple(g)) for g in boxes])
    rows.append((nm, step.mean() / diag, step.sum() / diag, static))

step_rel = np.array([r[1] for r in rows])
path_rel = np.array([r[2] for r in rows])
static = np.array([r[3] for r in rows])

print(f"across all {len(rows)} VOT sequences")
print(f"  distance travelled, in target widths : median {np.median(path_rel):.1f}")
print(f"  movement per frame, in target widths : median {np.median(step_rel):.3f}")
print(f"  a never-moving box scores            : mean {static.mean():.3f}")
print(f"  sequences where it exceeds 0.5       : {int((static > 0.5).sum())} of {len(rows)}")
print(f"  sequences where it falls below 0.1   : {int((static < 0.1).sum())} of {len(rows)}")

## 3. Sample sequences

Run the tracker on four sequences for inspection. Do not treat this small sample as the full benchmark.


In [ ]:
demo = ["bag", "blanket", "fernando", "basketball"]
print(
    f"{'sequence':>12}{'frames':>8}{'static':>9}{'tracker':>9}"
    f"{'succ@.5':>9}{'AUC':>8}{'centre err':>12}"
)
for nm in demo:
    e, b = DVSBenchmark(VOT, sequences=[nm])[0]
    x, y, t = e["x"], e["y"], e["t"]

    first = ev.gt_box(b[0])
    s_pred, s_true = [], []
    for s in np.arange(t.min(), t.max(), WINDOW):
        s_pred.append(first)
        s_true.append(ev.gt_at(b, s + WINDOW / 2))
    st = tracker.score(s_pred, s_true)

    p, g, _ = tracker.track(x, y, t, b, window=WINDOW, alpha=0.3)
    tr = tracker.score(p, g)
    print(
        f"{nm:>12}{tr['frames']:>8}{st['mean_iou']:>9.3f}{tr['mean_iou']:>9.3f}"
        f"{100 * tr['success_50']:>8.0f}%{tr['success_auc']:>8.3f}"
        f"{tr['median_centre_error']:>12.1f}"
    )

## 4. Recorded benchmark summary

The dictionaries below contain saved aggregate measurements, not a fresh benchmark run. Use bench_tracking.py to rerun the evaluation on local recordings.


In [ ]:
vot_full = {
    "static": 0.136,
    "tracker": 0.081,
    "wins": 17,
    "n": 60,
    "success50": 0.044,
    "above50": 0,
    "on_target": 0.20,
}
track_full = {
    "static": 0.206,
    "tracker": 0.103,
    "wins": 5,
    "n": 20,
    "success50": 0.057,
    "above50": 0,
    "on_target": 0.24,
}

print(f"{'':>26}{'VOT (60)':>12}{'Tracking (20)':>16}")
for label, key in [
    ("static box, mean IoU", "static"),
    ("local tracker, mean IoU", "tracker"),
    ("tracker success at 0.5", "success50"),
    ("events on target in window", "on_target"),
]:
    a, b = vot_full[key], track_full[key]
    fmt = (
        (lambda v: f"{100 * v:.0f}%")
        if key in ("success50", "on_target")
        else (lambda v: f"{v:.3f}")
    )
    print(f"{label:>26}{fmt(a):>12}{fmt(b):>16}")

for name, d in [("VOT", vot_full), ("TrackingDataset", track_full)]:
    print(
        f"\n{name}: tracker beats static on {d['wins']}/{d['n']} sequences, "
        f"{d['above50']}/{d['n']} reach 0.5 overlap"
    )
    print(f"     difference in mean IoU: {d['tracker'] - d['static']:+.3f}")

## 5. Target-event share

Only about a fifth of search-window activity was on the target in the recorded VOT evaluation. Activity-based selection can therefore prefer background regions even during local search.


## 6. Frame-window comparison

Compare recorded scores across frame windows, including monitor-refresh and source-video periods. The results do not support frame-window tuning as a sufficient remedy.


In [ ]:
sweep = {
    "16.7 ms  one 60 Hz refresh": 0.120,
    "33.3 ms  one 30 fps frame": 0.126,
    "66.7 ms  two video frames": 0.114,
    "100 ms   three video frames": 0.104,
    "10 ms    unaligned": 0.123,
    "25 ms    unaligned": 0.114,
    "50 ms    unaligned": 0.136,
}
static_8 = 0.119

print("eight VOT sequences, mean IoU against the annotations")
print(f"{'window':>30}{'IoU':>8}{'vs static':>12}")
print(f"{'static box, never moves':>30}{static_8:>8.3f}{'':>12}")
for k, v in sweep.items():
    print(f"{k:>30}{v:>8.3f}{v - static_8:>+12.3f}")

## 7. Annotation geometry

Annotations are rotated quadrilaterals; predictions are upright rectangles. Quantify the area increase from axis-aligned conversion. evaluate.py also estimates a fixed-size quadrilateral overlap ceiling.


In [ ]:
def quad_area(r):
    xs, ys = r[1::2], r[2::2]
    return 0.5 * abs(sum(xs[i] * ys[(i + 1) % 4] - xs[(i + 1) % 4] * ys[i] for i in range(4)))


def aabb_area(r):
    xs, ys = r[1::2], r[2::2]
    return (xs.max() - xs.min()) * (ys.max() - ys.min())


print(f"{'sequence':>14}{'true area':>11}{'as we score it':>16}{'inflation':>11}{'max IoU':>10}")
infl = []
for nm in ["ball1", "blanket", "bag", "basketball", "fernando", "gymnastics3"]:
    b = vot._boxes[nm]
    q = np.mean([quad_area(r) for r in b])
    a = np.mean([aabb_area(r) for r in b])
    infl.append(a / max(q, 1e-9))
    print(f"{nm:>14}{q:>11.0f}{a:>16.0f}{a / max(q, 1e-9):>11.2f}{max(q, 1e-9) / a:>10.2f}")
print(f"{'MEAN':>14}{'':>11}{'':>16}{np.mean(infl):>11.2f}{1 / np.mean(infl):>10.2f}")

## 8. Qualitative sequence

Plot the predicted and annotated boxes at six times in a sequence. Inspect misses and drift alongside aggregate IoU.


In [ ]:
SEQ = "blanket"
e, b = DVSBenchmark(VOT, sequences=[SEQ])[0]
x, y, t = e["x"], e["y"], e["t"]
preds, truths, times = tracker.track(x, y, t, b, window=WINDOW, alpha=0.3)

pick = np.linspace(0, len(preds) - 1, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4))
for ax, i in zip(axes.ravel(), pick):
    s = times[i]
    sel = (t >= s) & (t < s + WINDOW)
    img = np.zeros((180, 240), dtype=np.float32)
    np.add.at(img, (y[sel], x[sel]), 1.0)
    ax.imshow(img, cmap="hot")
    for box, colour in [(truths[i], "yellow"), (preds[i], "cyan")]:
        bx0, by0, bx1, by1 = box
        ax.add_patch(
            patches.Rectangle(
                (bx0, by0), bx1 - bx0, by1 - by0, lw=1.8, edgecolor=colour, facecolor="none"
            )
        )
    ax.set_title(
        f"t = {(s - t.min()) / 1e6:.1f} s   IoU {ev.iou(preds[i], truths[i]):.2f}", fontsize=9
    )
    ax.axis("off")
plt.suptitle(f"{SEQ}: annotation in yellow, prediction in cyan")
plt.tight_layout()
plt.show()

## 9. Conclusions

Use the full benchmark rather than a favourable sample. Report the static baseline, geometry constraints and target-event share. The appearance tracker in snn_tracker.py is a separate experiment.
